# RiskLens 03 · NeMo Gym: a verifiable environment for CECL investigations

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Luizkauffmann/Nvidia-Redhat-RiskLens/blob/main/notebooks/03_NeMo_Gym_Environment.ipynb)

[NeMo Gym](https://github.com/NVIDIA-NeMo/Gym) runs agents against **environments** and scores them with a **verifier**, for evaluation and for RL training. RiskLens is a natural fit: we own the simulator, so we know the right answer to every question.

| NeMo Gym concept | RiskLens |
|---|---|
| Dataset (JSONL, one task per row) | the 15 demo questions + paraphrases; each row carries the system prompt, the agent's tool schemas and `verifier_metadata` (expected facts from the seeded ground truth) |
| Resources server (tools + verifier) | `resources_servers/risklens_cecl/app.py`: all 30 deterministic tools as `POST /<tool>` endpoints, plus `/verify` |
| Agent harness | Gym's `simple_agent` (model ↔ tools loop) |
| Model server | Nemotron 3.5 Lightning on vLLM (local / Brev) or the NVIDIA API, through Gym's `vllm_model`; or our mock planner |

**Reward** (`config/gym.yaml`, each component in [0, 1]):
`0.55·correctness + 0.20·grounding + 0.10·tool_selection + 0.10·call_validity + 0.05·efficiency`, × 0.25 if the answer used no successful tool call (every episode replays the same story, so memorised answers must not pay), 0 with no answer.

**Local or Colab.** NeMo Gym needs Python ≥ 3.13.14. The setup cell installs it: into this environment if the Python is new enough, otherwise (e.g. Colab) into a separate Python 3.13 venv that only the `gym` command uses. The first server start builds one venv per Gym server (a few minutes); later starts take seconds. In Colab, use the `nvidia` policy or your Brev vLLM URL for section 5.

## 0 · Setup

In [ ]:
# RiskLens bootstrap: finds the repo checkout (local Jupyter) or clones it (Colab)
import os, sys, subprocess
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/Luizkauffmann/Nvidia-Redhat-RiskLens.git"

def _find_repo():
    env = os.environ.get("RISKLENS_REPO")
    for p in ([Path(env)] if env else []) + [Path.cwd(), *Path.cwd().parents]:
        if (p / "config" / "gym.yaml").exists() and (p / "src" / "tools").exists():
            return p.resolve()
    return None

REPO_DIR = _find_repo()
if REPO_DIR is None:
    if not IN_COLAB:
        raise SystemExit("Start Jupyter from the Nvidia-Redhat-RiskLens checkout (or set RISKLENS_REPO).")
    REPO_DIR = Path("/content/Nvidia-Redhat-RiskLens")
    if (REPO_DIR / ".git").exists():                 # runtime restarted: refresh the existing clone
        subprocess.run(["git", "-C", str(REPO_DIR), "pull", "-q", "--ff-only"], check=False)
    else:
        subprocess.run(["git", "clone", "-q", "--depth", "1", REPO_URL, str(REPO_DIR)], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(REPO_DIR / "requirements.txt"), "matplotlib"], check=True)

os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

from src.utils import config
config.reload()
from src.utils.notebook import M, ensure_demo_data, endpoint_up, load_secrets, secret
print("secrets:", load_secrets() or "none (.env locally, Colab Secrets in Colab)")
print("repo   :", REPO_DIR, "| python", sys.version.split()[0], "| colab" if IN_COLAB else "| local")
print("data   :", ensure_demo_data())

In [ ]:
import json, shutil
import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import Markdown, display
from src.evaluation import gym_runner, verifier
from src.evaluation.gym_dataset import build_rows, write_jsonl
from src.evaluation.local_eval import evaluate
from src.evaluation.trajectories import rollouts_to_sft
from src.llm.provider import get_provider

pd.set_option("display.max_colwidth", 90)
from src.utils.notebook import ensure_gym_cli
GYM = ensure_gym_cli()          # installs NeMo Gym if missing (a Python 3.13 venv when the kernel is older, e.g. Colab)
print("gym CLI:", GYM or "unavailable: sections 1-3 still run in-process")
if GYM:
    print(subprocess.run(["gym", "--version"], capture_output=True, text=True).stdout.splitlines()[0])

## 1 · The dataset
`build_rows()` turns the task bank into Gym rows. `example` = the 15 canonical demo questions, `train` = those plus paraphrases, `validation` = held-out paraphrases.

In [ ]:
splits = build_rows()
for name, rows in splits.items():
    write_jsonl(rows, gym_runner.DATA_DIR / f"{name}.jsonl")
    print(f"{name:<11}{len(rows):>3} rows -> {(gym_runner.DATA_DIR / f'{name}.jsonl').relative_to(REPO_DIR)}")

row = splits["example"][4]
p = row["responses_create_params"]
print("\nquestion :", p["input"][1]["content"])
print("tools    :", len(p["tools"]), "->", ", ".join(t["name"] for t in p["tools"][:6]), "...")
print("system   :", p["input"][0]["content"][:300].replace("\n", " "), "...")
pd.DataFrame(row["verifier_metadata"]["checks"])

## 2 · The reward function, by example
Three answers to *"By how many dollars did the allowance change?"*: a grounded one, one with invented numbers, and one that names the right figure without calling any tool.

In [ ]:
from src.tools import call_tool
task = next(r["verifier_metadata"] for r in splits["example"] if r["id"] == "q03_dollar_change")
rec = call_tool("compare_allowance", {"previous_date": "2026-07-31", "current_date": "2026-08-31"})
calls = [verifier.ToolCallTrace("compare_allowance", {}, {"status": "ok", "result": rec.result})]
c = rec.result
cases = {
    "grounded":     (f"The allowance increased by {M(c['change'])} ({c['change_pct']:.1f}%), from {M(c['previous_allowance'])} to {M(c['current_allowance'])}. [compare_allowance]", calls),
    "hallucinated": ("The allowance increased by $25.17M, driven by $412.9M of new loans.", calls),
    "no tools":     (f"The allowance increased by {M(c['change'])}.", []),
}
rows = []
for name, (answer, cs) in cases.items():
    res = verifier.score(answer, cs, task)
    rows.append({"case": name, "reward": res.reward, **res.components, "answer": answer})
pd.DataFrame(rows).set_index("case")

## 3 · Baseline in-process (no servers)
`evaluate()` runs each row with RiskLens's own agent loop and the same verifier. The mock planner is a keyword-matching script: a floor, not a model. Note where it breaks on paraphrases.

In [ ]:
mock_local = evaluate(get_provider({"provider": "mock"}), split="train")
print(json.dumps(mock_local.summary(), indent=1))
mock_local.frame().sort_values("reward")[["task_id", "reward", "c_correctness", "c_grounding", "tool_calls", "checks_passed"]].head(8)

## 4 · The same evaluation through real NeMo Gym servers
`run_gym_eval()` runs `gym env start` (resources server + `simple_agent` + `vllm_model` pointed at the policy), then `gym eval run --no-serve`, then stops the servers. For the mock policy it also starts `scripts/serve_policy.py`, an OpenAI-compatible endpoint, so the whole pipeline runs without a GPU.

Equivalent shell commands (from the repo root):
```bash
python -m scripts.serve_policy --provider mock --port 8011 &
gym env start --resources-server risklens_cecl --model-type vllm_model \
    --model mock-planner --model-url http://127.0.0.1:8011/v1 --model-api-key none
gym eval run --no-serve --agent risklens_cecl_simple_agent \
    --input resources_servers/risklens_cecl/data/train.jsonl --output data/outputs/gym/mock_train_rollouts.jsonl
```

In [ ]:
mock_path = gym_runner.run_gym_eval("mock", split="train", stream=False)
mock_gym = gym_runner.load_rollouts(mock_path)
print(gym_runner.summarize(mock_gym))
parity = mock_gym.set_index("task_id")["reward"].to_frame("gym").join(mock_local.frame().set_index("task_id")["reward"].rename("in_process"))
print("max |gym - in-process| reward difference:", float((parity.gym - parity.in_process).abs().max()))
mock_gym[["task_id", "reward", "tool_calls", "tools", "failed_checks"]]

## 5 · Nemotron 3.5 Lightning in the environment
Point Gym's model server at Nemotron. `vllm` = local or Brev vLLM (`deploy/brev_serve_nemotron.sh`), `nvidia` = NVIDIA API catalog. `NUM_REPEATS > 1` samples each task several times, which shows how consistent the model is (and is what RL needs).

In [ ]:
POLICY = os.environ.get("RISKLENS_GYM_POLICY", "auto")      # auto | vllm | nvidia | mock
NUM_REPEATS = int(os.environ.get("RISKLENS_GYM_REPEATS", "2"))
SPLIT = "example"

vllm = gym_runner.POLICIES["vllm"]
if POLICY == "auto":
    POLICY = "vllm" if endpoint_up(vllm.url) else ("nvidia" if secret("NVIDIA_API_KEY") else None)
if POLICY == "nvidia":
    os.environ["NVIDIA_API_KEY"] = secret("NVIDIA_API_KEY") or ""
print("policy:", POLICY or "none reachable: start vLLM or add NVIDIA_API_KEY to .env, then re-run this cell")

results = {"mock": mock_gym}
if POLICY:
    path = gym_runner.run_gym_eval(POLICY, split=SPLIT, num_repeats=NUM_REPEATS, stream=False)
    results[POLICY] = gym_runner.load_rollouts(path)
    print(gym_runner.summarize(results[POLICY]))
    display(results[POLICY].groupby("task_id")[["reward", "correctness", "grounding", "tool_calls"]].mean().sort_values("reward"))

## 6 · Compare policies

In [ ]:
comps = ["reward", "correctness", "grounding", "tool_selection", "call_validity", "efficiency"]
table = pd.DataFrame({name: df[df.task_id.isin([r["id"] for r in splits["example"]])][comps].mean() for name, df in results.items()}).T
display(table.round(3))
ax = table.T.plot.bar(rot=20, title="NeMo Gym reward components on the 15 demo questions", figsize=(10, 4))
ax.set_ylim(0, 1.05); plt.tight_layout(); plt.show()
if len(results) > 1:
    by_cat = pd.concat({n: df.groupby("category")["reward"].mean() for n, df in results.items()}, axis=1)
    display(by_cat.round(3))

## 7 · Inspect a rollout
What the policy actually did: tool calls in order, the verifier's checks and the final answer.

In [ ]:
name = next(iter(n for n in results if n != "mock"), "mock")
df = results[name]
worst = df.sort_values("reward").iloc[0]
print(f"[{name}] {worst.task_id}  reward={worst.reward:.3f}  tools={worst.tools}")
print("failed checks:", worst.failed_checks or "none")
display(Markdown(worst.answer or "_(no answer)_"))

## 8 · From rollouts to training data
Rollouts that pass the verifier (reward ≥ 0.8) become supervised fine-tuning examples in OpenAI chat format with tool calls: grounded, correct investigations only. This is the hand-off to LoRA SFT (Fine-Tuned track) and, with the same resources server as the reward, to RL with NeMo RL (GRPO) on Brev.

In [ ]:
paths = sorted((REPO_DIR / "data" / "outputs" / "gym").glob("*_rollouts.jsonl"))
sft_path = REPO_DIR / "data" / "outputs" / "gym" / "sft_trajectories.jsonl"
sft = rollouts_to_sft([p for p in paths if not p.name.startswith("mock")] or paths, min_reward=0.8, out=sft_path)
print(f"{len(sft)} examples -> {sft_path.relative_to(REPO_DIR)}  (from {[p.name for p in paths]})")
if sft:
    ex = sft[0]
    print("roles:", [m["role"] for m in ex["messages"]], "| task:", ex["metadata"]["task_id"], "| reward:", ex["metadata"]["reward"])

**Next steps**
- `python -m scripts.build_gym_dataset --agent credit_agent` builds the same environment with a specialist's 15-tool allow-list, so each Phase 4 specialist can be evaluated (and later trained) on its own slice.
- Add story variants (credit-led, growth-led, a real model change) to the generator so the policy has to investigate rather than memorise; each variant ships its own ground truth and Gym rows.
- RL: NeMo RL's GRPO recipes consume a Gym resources server as the reward. Full RL on Nemotron 3.5 Lightning needs far more GPUs than a hackathon box, so the weekend deliverable is the environment, the baseline scores and the SFT set.